In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("cwiczenia/RTA/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [3]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
# 1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
from pyspark.sql.functions import window, avg, round as _round
hourly = (
    df.groupBy(window("timestamp", "1 hour"), "store")    # okno 1-godzinne
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .orderBy("window")
)
hourly[hourly["store"] == "Gdańsk"].orderBy("srednia_PLN").show(truncate=False, n=1)

+------------------------------------------+------+-----------+
|window                                    |store |srednia_PLN|
+------------------------------------------+------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Gdańsk|395.01     |
+------------------------------------------+------+-----------+
only showing top 1 row



In [5]:
# 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.
from pyspark.sql.functions import count
category_summary = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("category")
)

category_summary[category_summary["window"].start == "2026-04-12 09:00:00"].show()

+--------------------+-----------+---------+
|              window|   category|liczba_tx|
+--------------------+-----------+---------+
|{2026-04-12 09:00...|elektronika|      611|
|{2026-04-12 09:00...|    książki|      622|
|{2026-04-12 09:00...|     odzież|      605|
|{2026-04-12 09:00...|    żywność|      567|
+--------------------+-----------+---------+



In [6]:
# 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
quarterly = (
    df.groupBy(window("timestamp", "15 minutes"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("liczba_tx", ascending = False)
)
quarterly.show(truncate=False, n=1)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+
only showing top 1 row



In [7]:
spark.stop()